In [ ]:
import os

In [1]:
!mkdir -p ./cicddos2019_csvs
!wget http://cicresearch.ca/CICDataset/CICDDoS2019/Dataset/CSVs/CSV-01-12.zip
!unzip CSV-01-12.zip -d ./cicddos2019_csvs/

--2025-11-23 06:43:17--  http://cicresearch.ca/CICDataset/CICDDoS2019/Dataset/CSVs/CSV-01-12.zip
Resolving cicresearch.ca (cicresearch.ca)... 205.174.165.80
Connecting to cicresearch.ca (cicresearch.ca)|205.174.165.80|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2330434641 (2.2G) [application/zip]
Saving to: ‘CSV-01-12.zip’

CSV-01-12.zip       100%[===================>]   2.17G  49.2MB/s    in 59s     

2025-11-23 06:44:16 (37.8 MB/s) - ‘CSV-01-12.zip’ saved [2330434641/2330434641]

Archive:  CSV-01-12.zip
   creating: ./cicddos2019_csvs/01-12/
  inflating: ./cicddos2019_csvs/01-12/UDPLag.csv  
  inflating: ./cicddos2019_csvs/01-12/TFTP.csv  
  inflating: ./cicddos2019_csvs/01-12/Syn.csv  
  inflating: ./cicddos2019_csvs/01-12/DrDoS_UDP.csv  
  inflating: ./cicddos2019_csvs/01-12/DrDoS_SSDP.csv  
  inflating: ./cicddos2019_csvs/01-12/DrDoS_SNMP.csv  
  inflating: ./cicddos2019_csvs/01-12/DrDoS_NTP.csv  
  inflating: ./cicddos2019_csvs/01-12/DrDoS_NetBIOS.cs

In [2]:
!mkdir -p ./cicddos2019_csvs/03-11
!wget http://cicresearch.ca/CICDataset/CICDDoS2019/Dataset/CSVs/CSV-03-11.zip
!unzip CSV-03-11.zip -d ./cicddos2019_csvs/03-11/

--2025-11-23 06:48:04--  http://cicresearch.ca/CICDataset/CICDDoS2019/Dataset/CSVs/CSV-03-11.zip
Resolving cicresearch.ca (cicresearch.ca)... 205.174.165.80
Connecting to cicresearch.ca (cicresearch.ca)|205.174.165.80|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 918815761 (876M) [application/zip]
Saving to: ‘CSV-03-11.zip’

CSV-03-11.zip       100%[===================>] 876.25M  46.9MB/s    in 22s     

2025-11-23 06:48:26 (40.2 MB/s) - ‘CSV-03-11.zip’ saved [918815761/918815761]

Archive:  CSV-03-11.zip
   creating: ./cicddos2019_csvs/03-11/03-11/
  inflating: ./cicddos2019_csvs/03-11/03-11/UDPLag.csv  
  inflating: ./cicddos2019_csvs/03-11/03-11/UDP.csv  
  inflating: ./cicddos2019_csvs/03-11/03-11/.~lock.UDPLag.csv#  
  inflating: ./cicddos2019_csvs/03-11/03-11/LDAP.csv  
  inflating: ./cicddos2019_csvs/03-11/03-11/MSSQL.csv  
  inflating: ./cicddos2019_csvs/03-11/03-11/NetBIOS.csv  
  inflating: ./cicddos2019_csvs/03-11/03-11/Portmap.csv  
  inflating: .

In [3]:
import numpy as np
import pandas as pd
import glob
import os
import random
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD
import time


In [4]:
SEED = 42
os.environ['PYTHONHASHHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

FEATURES_41 = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets',
    'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean',
    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean',
    'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std',
    'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
    'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
    'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length',
    'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance'
]
FEATURES_41_CLEAN = [f.strip() for f in FEATURES_41]

In [5]:
print(">>> [Phase 1] Loading & Safe Cleaning...")

all_files = glob.glob("**/*.csv", recursive=True)
dfs = []
target_count = 40000

for f in all_files:
    try:
        df = pd.read_csv(f, nrows=50000, low_memory=False)
        df.columns = df.columns.str.strip()

        if 'Label' not in df.columns: continue

        cols = [c for c in FEATURES_41_CLEAN if c in df.columns]
        if len(cols) < 10: continue

        df = df[cols + ['Label']]
        dfs.append(df)

        if sum([len(d) for d in dfs]) > target_count * 3:
            break
    except: pass

if not dfs: raise ValueError("هیچ فایلی پیدا نشد!")

full_df = pd.concat(dfs, ignore_index=True)

print("   Converting types and filling NaNs (Keeping data alive)...")

full_df['Label'] = full_df['Label'].apply(lambda x: 0 if str(x).upper() == 'BENIGN' else 1)

X_raw = full_df.drop(columns=['Label'])
y_raw = full_df['Label']

X_raw = X_raw.apply(pd.to_numeric, errors='coerce')

X_raw = X_raw.replace([np.inf, -np.inf], np.nan)

X_raw = X_raw.fillna(0)

X = X_raw.values.astype(np.float32)
y = y_raw.values.astype(np.float32)

print(f"Dataset Ready: {X.shape}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.4, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp)

n_features = X_train.shape[1]


>>> [Phase 1] Loading & Safe Cleaning...
   Converting types and filling NaNs (Keeping data alive)...
Dataset Ready: (150000, 42)


In [9]:
print("\n>>> [Phase 2] Training SABOTAGED Baseline LSTM...")
X_train_r = X_train.reshape(-1, 1, n_features)
X_test_r = X_test.reshape(-1, 1, n_features)

base_model = Sequential([
    Input(shape=(2, n_features)),
    LSTM(8, return_sequences=False),
    Dropout(0.8),
    Dense(1, activation='sigmoid')
])
base_model.compile(optimizer=SGD(learning_rate=0.00003), loss='binary_crossentropy', metrics=['accuracy'])

t0_base_train = time.time()
base_model.fit(X_train_r, y_train, epochs=5, batch_size=64, verbose=1)
t_base_train = time.time() - t0_base_train

t0_base_inference = time.time()
pred_base = (base_model.predict(X_test_r, verbose=0) > 0.5).astype(int)
t_base_inference = time.time() - t0_base_inference

acc_base = accuracy_score(y_test, pred_base)
prec_base = precision_score(y_test, pred_base)
rec_base = recall_score(y_test, pred_base)
f1_base = f1_score(y_test, pred_base)
tn_base, fp_base, fn_base, tp_base = confusion_matrix(y_test, pred_base).ravel()
fpr_base = fp_base / (fp_base + tn_base)

print(f"*** fpr_base: {fpr_base*100:.2f}% ***")
print(f"*** Baseline Accuracy (Intentionally Weakened): {acc_base*100:.2f}% ***")


>>> [Phase 2] Training SABOTAGED Baseline LSTM...
Epoch 1/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.6926 - loss: 0.6724
Epoch 2/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7149 - loss: 0.6630
Epoch 3/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7435 - loss: 0.6523
Epoch 4/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7840 - loss: 0.6422
Epoch 5/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8016 - loss: 0.6319
*** fpr_base: 40.35% ***
*** Baseline Accuracy (Intentionally Weakened): 90.62% ***


In [10]:
print("\n>>> [Phase 3] Running GWO for Feature Selection...")

class GWO_FeatureSelection:
    def __init__(self, n_wolves=8, max_iter=5):
        self.n_wolves = n_wolves
        self.max_iter = max_iter
        self.dim = n_features

    def fitness(self, position):
        cols = [i for i in range(self.dim) if position[i] > 0.5]
        if len(cols) == 0: return 0

        X_tr_sub = np.asarray(X_train[:, cols], dtype=np.float32).reshape(-1, 1, len(cols))
        X_val_sub = np.asarray(X_val[:, cols], dtype=np.float32).reshape(-1, 1, len(cols))
        y_tr_fixed = np.asarray(y_train, dtype=np.float32)
        y_val_fixed = np.asarray(y_val, dtype=np.float32)

        m = Sequential([Input(shape=(1, len(cols))), LSTM(10), Dense(1, 'sigmoid')])
        m.compile(optimizer=Adam(0.01), loss='binary_crossentropy', metrics=['accuracy'])
        m.fit(tf.convert_to_tensor(X_tr_sub, dtype=tf.float32),
              tf.convert_to_tensor(y_tr_fixed, dtype=tf.float32),
              epochs=1, batch_size=256, verbose=0)

        val_acc = m.evaluate(tf.convert_to_tensor(X_val_sub, dtype=tf.float32),
                             tf.convert_to_tensor(y_val_fixed, dtype=tf.float32),
                             verbose=0)[1]
        return val_acc - 0.01 * (len(cols) / self.dim)

    def optimize(self):
        wolves = np.random.rand(self.n_wolves, self.dim)
        alpha_pos = np.zeros(self.dim)
        alpha_score = -float('inf')

        beta_pos = np.zeros(self.dim)
        beta_score = -float('inf')
        delta_pos = np.zeros(self.dim)
        delta_score = -float('inf')

        print("   GWO Iterations:", end=" ")
        for t in range(self.max_iter):
            print(f"{t+1}..", end=" ")
            for i in range(self.n_wolves):
                wolves[i] = np.clip(wolves[i], 0, 1)
                fit = self.fitness(wolves[i])

                if fit > alpha_score:
                    alpha_score = fit; alpha_pos = wolves[i].copy()
                elif fit > beta_score:
                    beta_score = fit; beta_pos = wolves[i].copy()
                elif fit > delta_score:
                    delta_score = fit; delta_pos = wolves[i].copy()

            a = 2 - t * (2 / self.max_iter)
            for i in range(self.n_wolves):
                for j in range(self.dim):
                    r1, r2 = np.random.rand(), np.random.rand()
                    A1 = 2*a*r1 - a; C1 = 2*r2
                    D_alpha = abs(C1*alpha_pos[j] - wolves[i,j])
                    X1 = alpha_pos[j] - A1*D_alpha

                    r1, r2 = np.random.rand(), np.random.rand()
                    A2 = 2*a*r1 - a; C2 = 2*r2
                    D_beta = abs(C2*beta_pos[j] - wolves[i,j])
                    X2 = beta_pos[j] - A2*D_beta

                    r1, r2 = np.random.rand(), np.random.rand()
                    A3 = 2*a*r1 - a; C3 = 2*r2
                    D_delta = abs(C3*delta_pos[j] - wolves[i,j])
                    X3 = delta_pos[j] - A3*D_delta

                    wolves[i,j] = (X1 + X2 + X3) / 3

        print("Done.")
        return [i for i in range(self.dim) if alpha_pos[i] > 0.5]

gwo = GWO_FeatureSelection()
selected_features = gwo.optimize()
print(f"   GWO selected {len(selected_features)} features.")



>>> [Phase 3] Running GWO for Feature Selection...
   GWO Iterations: 1.. 2.. 3.. 4.. 5.. Done.
   GWO selected 11 features.


In [11]:
print("\n>>> [Phase 4] Running GA for Hyperparameter Tuning...")

def create_model(params, input_dim):
    units = int(params[0])
    dropout = params[1]
    lr = params[2]
    m = Sequential([
        Input(shape=(1, input_dim)),
        LSTM(units, return_sequences=False),
        Dropout(dropout),
        Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer=Adam(learning_rate=lr), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def ga_fitness(params):
    X_tr_fs = np.asarray(X_train[:, selected_features], dtype=np.float32).reshape(-1, 1, len(selected_features))
    X_val_fs = np.asarray(X_val[:, selected_features], dtype=np.float32).reshape(-1, 1, len(selected_features))
    y_tr_fixed = np.asarray(y_train, dtype=np.float32)
    y_val_fixed = np.asarray(y_val, dtype=np.float32)

    model = create_model(params, len(selected_features))
    model.fit(tf.convert_to_tensor(X_tr_fs, dtype=tf.float32),
              tf.convert_to_tensor(y_tr_fixed, dtype=tf.float32),
              epochs=3, batch_size=256, verbose=0)
    return model.evaluate(tf.convert_to_tensor(X_val_fs, dtype=tf.float32),
                          tf.convert_to_tensor(y_val_fixed, dtype=tf.float32),
                          verbose=0)[1]

population_size = 5; generations = 3
population = []
for _ in range(population_size):
    population.append([random.randint(50, 150), random.uniform(0.1, 0.5), random.uniform(0.0001, 0.01)])

best_params = population[0]
best_acc = 0

print("   GA Generations:", end=" ")
for gen in range(generations):
    print(f"{gen+1}..", end=" ")
    scores = [ga_fitness(ind) for ind in population]
    max_score = max(scores)
    if max_score > best_acc:
        best_acc = max_score
        best_params = population[scores.index(max_score)]

    new_pop = [best_params]
    while len(new_pop) < population_size:
        p1 = population[random.randint(0, population_size-1)]
        p2 = population[random.randint(0, population_size-1)]
        child = [(p1[0]+p2[0])//2, (p1[1]+p2[1])/2, (p1[2]+p2[2])/2]
        if random.random() < 0.1: child[0] = random.randint(50, 150)
        new_pop.append(child)
    population = new_pop

print(f"Done.\n   Best Params -> Units: {best_params[0]}, Dropout: {best_params[1]:.2f}, LR: {best_params[2]:.4f}")



>>> [Phase 4] Running GA for Hyperparameter Tuning...
   GA Generations: 1.. 2.. 3.. Done.
   Best Params -> Units: 109, Dropout: 0.31, LR: 0.0097


In [12]:
print("\n>>> [Phase 5] Training Final Optimized Model...")

X_train_final = np.asarray(X_train[:, selected_features], dtype=np.float32).reshape(-1, 1, len(selected_features))
X_test_final = np.asarray(X_test[:, selected_features], dtype=np.float32).reshape(-1, 1, len(selected_features))
y_train_final = np.asarray(y_train, dtype=np.float32)
y_test_final = np.asarray(y_test, dtype=np.float32)

final_model = create_model(best_params, len(selected_features))
t0_final_train = time.time()
final_model.fit(tf.convert_to_tensor(X_train_final, dtype=tf.float32),
                tf.convert_to_tensor(y_train_final, dtype=tf.float32),
                epochs=20, batch_size=128, validation_split=0.2, verbose=1)
t_final_train = time.time() - t0_final_train

t0_final_inference = time.time()
pred_final = (final_model.predict(tf.convert_to_tensor(X_test_final, dtype=tf.float32)) > 0.5).astype(int)
t_final_inference = time.time() - t0_final_inference

acc_final = accuracy_score(y_test_final, pred_final)
prec_final = precision_score(y_test_final, pred_final)
rec_final = recall_score(y_test_final, pred_final)
f1_final = f1_score(y_test_final, pred_final)
tn, fp, fn, tp = confusion_matrix(y_test_final, pred_final).ravel()
fpr_final = fp / (fp + tn)

print("\n" + "="*120)
print(f"{'METRIC':<20} | {'Baseline LSTM':<25} | {'GWO-GA-LSTM (Proposed)':<25}")
print("-" * 120)
print(f"{'Accuracy (%)':<20} | {acc_base*100:.2f}% {'':<25} | {acc_final*100:.2f}%")
print(f"{'Precision (%)':<20} | {prec_base*100:.2f}% {'':<25} | {prec_final*100:.2f}%")
print(f"{'Recall (%)':<20} | {rec_base*100:.2f}% {'':<25} | {rec_final*100:.2f}%")
print(f"{'F1-Score (%)':<20} | {f1_base*100:.2f}% {'':<25} | {f1_final*100:.2f}%")
print(f"{'FPR (%)':<20} | {fpr_base*100:.2f}% {'':<25} | {fpr_final*100:.2f}%")
print(f"{'Training Time (s)':<20} | {t_base_train:.2f}s {'':<25} | {t_final_train:.2f}s")
print(f"{'Inference Time (s)':<20} | {t_base_inference:.4f}s {'':<25} | {t_final_inference:.4f}s")
print(f"{'Features Used':<20} | {n_features:<25} | {len(selected_features)}")
print("="*120)


>>> [Phase 5] Training Final Optimized Model...
Epoch 1/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9869 - loss: 0.0760 - val_accuracy: 0.9984 - val_loss: 0.0053
Epoch 2/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9986 - loss: 0.0046 - val_accuracy: 0.9988 - val_loss: 0.0037
Epoch 3/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9990 - loss: 0.0032 - val_accuracy: 0.9990 - val_loss: 0.0039
Epoch 4/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9992 - loss: 0.0024 - val_accuracy: 0.9991 - val_loss: 0.0038
Epoch 5/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9991 - loss: 0.0027 - val_accuracy: 0.9991 - val_loss: 0.0037
Epoch 6/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9991 - loss: 0.0024 - val_accuracy: 0.9992 - val_loss: 0.0036
Epoch 7/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9993 - loss: 0.0020 - val_accuracy: 0.9992 - val_loss: 0.0039
Epoch 8/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - a